In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
# 1. LOAD AND PREPROCESS DATA
df = pd.read_csv('creditcard.csv')
# Separate features (X) and target label (y)
X = df.drop(columns=['Class'])
y = df['Class']
# Split into training and testing sets (stratified because fraud is highly imbalanced)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.67, random_state=45, stratify=y
)
# Scale features (Crucial for LDA)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
# 3. APPLY LDA (Supervised Dimensionality Reduction)
lda = LDA(n_components=1)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)  # Requires y_train
X_test_lda = lda.transform(X_test_scaled)

In [4]:
# 4. EVALUATE AND COMPARE BOTH APPROACHES
def evaluate_model(X_tr, X_te, name):
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_tr, y_train)
    predictions = model.predict(X_te)
    probs = model.predict_proba(X_te)[:, 1]
    
    print(f"=== Performance using {name} Components ===")
    print(classification_report(y_test, predictions))
    print(f"ROC AUC Score: {roc_auc_score(y_test, probs):.4f}\n")

# Run evaluations
evaluate_model(X_train_lda, X_test_lda, "LDA (1 component)")

=== Performance using LDA (1 component) Components ===
              precision    recall  f1-score   support

           0       1.00      0.98      0.99    190491
           1       0.07      0.91      0.13       330

    accuracy                           0.98    190821
   macro avg       0.54      0.95      0.56    190821
weighted avg       1.00      0.98      0.99    190821

ROC AUC Score: 0.9842

